# Step 3 — Text Preprocessing Pipeline
**Project:** SDG Mapping for PhD Thesis  
**Purpose:** Clean and normalise text for all downstream steps — TF-IDF, embeddings, and BERT  
**Inputs:** `data/clean/X_train.csv` · `data/clean/X_test.csv` · `data/clean/mahe_corpus.csv`  
**Outputs:** `data/clean/X_train_processed.csv` · `data/clean/X_test_processed.csv` · `data/clean/mahe_processed.csv`  

> Each preprocessing variant is saved separately so Steps 4–7 can each use the right version:  
> - **TF-IDF / classical ML** → lemmatized, stopwords removed  
> - **Word2Vec / GloVe** → lightly cleaned, stopwords kept (embeddings need context)  
> - **BERT** → raw cleaned text only (BERT has its own tokenizer)


In [1]:
# Cell 1: Install dependencies 
!pip install nltk pandas numpy tqdm


In [2]:
# Cell 2: Imports & NLTK downloads 
import re
import json
import nltk
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from IPython.display import display

tqdm.pandas()

# Download NLTK resources
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

CLEAN_DIR = Path("data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

lemmatizer  = WordNetLemmatizer()
STOP_WORDS  = set(stopwords.words("english"))

# Domain stopwords — common academic words that add no SDG signal
DOMAIN_STOPS = {
    "study", "analysis", "result", "paper", "research", "review",
    "method", "approach", "using", "based", "among", "effect",
    "also", "may", "however", "used", "use", "associated",
    "significant", "significantly", "observed", "found", "show",
    "showed", "conclusion", "objective", "aim", "background",
    "introduction", "purpose", "data", "model", "evaluate",
    "evaluation", "performance", "proposed", "present", "case",
    "provide", "patient", "sample", "group", "level", "high",
    "low", "increase", "decrease", "compared", "compare",
    "total", "number", "rate", "value", "factor", "two",
    "three", "one", "new", "different", "large", "small"
}

ALL_STOPS = STOP_WORDS | DOMAIN_STOPS

print(f"✅ NLTK ready")
print(f"   Base stopwords    : {len(STOP_WORDS)}")
print(f"   Domain stopwords  : {len(DOMAIN_STOPS)}")
print(f"   Combined stopwords: {len(ALL_STOPS)}")


✅ NLTK ready
   Base stopwords    : 198
   Domain stopwords  : 61
   Combined stopwords: 259


In [3]:
# Cell 3: Preprocessing functions 

def clean_basic(text):
    """
    Basic cleaning shared by all variants.
    Removes URLs, emails, numbers, special characters.
    Keeps only alphabetic content.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # remove URLs
    text = re.sub(r"\S+@\S+", " ", text)                  # remove emails
    text = re.sub(r"\d+", " ", text)                       # remove numbers
    text = re.sub(r"[^a-z\s]", " ", text)                  # keep only letters
    text = re.sub(r"\s+", " ", text).strip()               # collapse whitespace
    return text


def preprocess_tfidf(text):
    """
    For TF-IDF and classical ML classifiers.
    Full pipeline: clean → tokenize → remove stopwords → lemmatize.
    Produces the most compressed, signal-dense representation.
    """
    text = clean_basic(text)
    if not text:
        return ""
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t.isalpha() and len(t) > 2 and t not in ALL_STOPS
    ]
    return " ".join(tokens)


def preprocess_embeddings(text):
    """
    For Word2Vec and GloVe embeddings.
    Light cleaning only — keep stopwords so embeddings
    learn proper word context and co-occurrence patterns.
    """
    text = clean_basic(text)
    if not text:
        return ""
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and len(t) > 1]
    return " ".join(tokens)


def preprocess_bert(text):
    """
    For BERT / Sentence-BERT.
    Minimal cleaning only — BERT has its own WordPiece tokenizer
    and handles casing, stopwords, and punctuation internally.
    Do NOT lemmatize or remove stopwords for BERT.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    # Truncate to 512 chars (BERT max token limit is ~512 tokens)
    return text[:512]


print("✅ Preprocessing functions defined")
print()
print("Three variants:")
print("  preprocess_tfidf()       → lemmatized, stopwords removed")
print("  preprocess_embeddings()  → lightly cleaned, stopwords kept")
print("  preprocess_bert()        → minimal cleaning, BERT handles the rest")


✅ Preprocessing functions defined

Three variants:
  preprocess_tfidf()       → lemmatized, stopwords removed
  preprocess_embeddings()  → lightly cleaned, stopwords kept
  preprocess_bert()        → minimal cleaning, BERT handles the rest


In [4]:
# Cell 4: Spot-check preprocessing on sample texts 
samples = [
    "Effectiveness of a community-based health intervention on reducing malnutrition among children under five in rural Karnataka",
    "Solar energy storage systems and grid integration challenges for renewable electricity generation",
    "Gender-based violence prevention programs and women empowerment in urban communities",
]

print("=" * 70)
print("PREPROCESSING SPOT CHECK")
print("=" * 70)

for i, text in enumerate(samples, 1):
    print(f"\n[Sample {i}]")
    print(f"  Original   : {text}")
    print(f"  TF-IDF     : {preprocess_tfidf(text)}")
    print(f"  Embeddings : {preprocess_embeddings(text)}")
    print(f"  BERT       : {preprocess_bert(text)}")
    print()

print("Key differences:")
print("  TF-IDF removes stopwords + lemmatizes → shortest, most compressed")
print("  Embeddings keeps full words → richer context for Word2Vec/GloVe")
print("  BERT keeps original text → tokenizer handles everything internally")


PREPROCESSING SPOT CHECK

[Sample 1]
  Original   : Effectiveness of a community-based health intervention on reducing malnutrition among children under five in rural Karnataka
  TF-IDF     : effectiveness community health intervention reducing malnutrition child five rural karnataka
  Embeddings : effectiveness of community based health intervention on reducing malnutrition among children under five in rural karnataka
  BERT       : Effectiveness of a community-based health intervention on reducing malnutrition among children under five in rural Karnataka


[Sample 2]
  Original   : Solar energy storage systems and grid integration challenges for renewable electricity generation
  TF-IDF     : solar energy storage system grid integration challenge renewable electricity generation
  Embeddings : solar energy storage systems and grid integration challenges for renewable electricity generation
  BERT       : Solar energy storage systems and grid integration challenges for renewable elect

In [5]:
# Cell 5: Load split datasets 
X_train = pd.read_csv(CLEAN_DIR / "X_train.csv")["text"]
X_test  = pd.read_csv(CLEAN_DIR / "X_test.csv")["text"]
df_mahe = pd.read_csv(CLEAN_DIR / "mahe_corpus.csv")

print(f"X_train : {len(X_train):,} rows")
print(f"X_test  : {len(X_test):,} rows")
print(f"MAHE    : {len(df_mahe):,} rows")


X_train : 27,204 rows
X_test  : 6,801 rows
MAHE    : 854 rows


In [6]:
# Cell 6: Apply TF-IDF preprocessing
print("Applying TF-IDF preprocessing...")

X_train_tfidf = X_train.progress_apply(preprocess_tfidf)
X_test_tfidf  = X_test.progress_apply(preprocess_tfidf)
mahe_tfidf    = df_mahe["text"].progress_apply(preprocess_tfidf)

# Check empty texts after processing
train_empty = (X_train_tfidf.str.strip() == "").sum()
test_empty  = (X_test_tfidf.str.strip() == "").sum()
mahe_empty  = (mahe_tfidf.str.strip() == "").sum()

print(f"\nTF-IDF preprocessing complete:")
print(f"  X_train empty after processing : {train_empty}")
print(f"  X_test  empty after processing : {test_empty}")
print(f"  MAHE    empty after processing : {mahe_empty}")

if train_empty > 0 or test_empty > 0:
    print("  ⚠ Some texts became empty — will replace with original cleaned text")
    X_train_tfidf = X_train_tfidf.where(X_train_tfidf.str.strip() != "", X_train.apply(clean_basic))
    X_test_tfidf  = X_test_tfidf.where(X_test_tfidf.str.strip()  != "", X_test.apply(clean_basic))

print("\nSample output (TF-IDF):")
for orig, proc in zip(X_train.head(3), X_train_tfidf.head(3)):
    print(f"  Before: {orig[:80]}...")
    print(f"  After : {proc[:80]}...")
    print()


Applying TF-IDF preprocessing...


  0%|          | 0/27204 [00:00<?, ?it/s]

  0%|          | 0/6801 [00:00<?, ?it/s]

  0%|          | 0/854 [00:00<?, ?it/s]


TF-IDF preprocessing complete:
  X_train empty after processing : 0
  X_test  empty after processing : 0
  MAHE    empty after processing : 0

Sample output (TF-IDF):
  Before: Excellent health care will be central to achieving this. Thus far, Japanese heal...
  After : excellent health care central achieving thus far japanese health care performed ...

  Before: Established conflict theories focus on the role of incentives in the decision to...
  After : established conflict theory focus role incentive decision join stay leave insurg...

  Before: Collective bargaining mechanisms were first piloted in the mid-1990s and have be...
  After : collective bargaining mechanism first piloted mid evolving ever since landmark l...



In [7]:
# Cell 7: Apply embedding preprocessing 
print("Applying embedding preprocessing...")

X_train_emb = X_train.progress_apply(preprocess_embeddings)
X_test_emb  = X_test.progress_apply(preprocess_embeddings)
mahe_emb    = df_mahe["text"].progress_apply(preprocess_embeddings)

print(f"\nEmbedding preprocessing complete")
print(f"\nSample output (Embeddings):")
for orig, proc in zip(X_train.head(2), X_train_emb.head(2)):
    print(f"  Before: {orig[:80]}...")
    print(f"  After : {proc[:80]}...")
    print()


Applying embedding preprocessing...


  0%|          | 0/27204 [00:00<?, ?it/s]

  0%|          | 0/6801 [00:00<?, ?it/s]

  0%|          | 0/854 [00:00<?, ?it/s]


Embedding preprocessing complete

Sample output (Embeddings):
  Before: Excellent health care will be central to achieving this. Thus far, Japanese heal...
  After : excellent health care will be central to achieving this thus far japanese health...

  Before: Established conflict theories focus on the role of incentives in the decision to...
  After : established conflict theories focus on the role of incentives in the decision to...



In [8]:
# Cell 8: Apply BERT preprocessing 
print("Applying BERT preprocessing...")

X_train_bert = X_train.progress_apply(preprocess_bert)
X_test_bert  = X_test.progress_apply(preprocess_bert)
mahe_bert    = df_mahe["text"].progress_apply(preprocess_bert)

print(f"\nBERT preprocessing complete")
print(f"\nSample output (BERT — minimal cleaning):")
for orig, proc in zip(X_train.head(2), X_train_bert.head(2)):
    print(f"  Before: {orig[:80]}...")
    print(f"  After : {proc[:80]}...")
    print()


Applying BERT preprocessing...


  0%|          | 0/27204 [00:00<?, ?it/s]

  0%|          | 0/6801 [00:00<?, ?it/s]

  0%|          | 0/854 [00:00<?, ?it/s]


BERT preprocessing complete

Sample output (BERT — minimal cleaning):
  Before: Excellent health care will be central to achieving this. Thus far, Japanese heal...
  After : Excellent health care will be central to achieving this. Thus far, Japanese heal...

  Before: Established conflict theories focus on the role of incentives in the decision to...
  After : Established conflict theories focus on the role of incentives in the decision to...



In [9]:
# Cell 9: Vocabulary analysis 
from collections import Counter

# Vocabulary size comparison
all_tfidf_tokens = " ".join(X_train_tfidf.tolist()).split()
all_emb_tokens   = " ".join(X_train_emb.tolist()).split()

vocab_tfidf = len(set(all_tfidf_tokens))
vocab_emb   = len(set(all_emb_tokens))

print("=== Vocabulary Analysis ===")
print(f"TF-IDF variant vocab size  : {vocab_tfidf:,} unique tokens")
print(f"Embedding variant vocab    : {vocab_emb:,} unique tokens")
print(f"Vocab reduction            : {(1 - vocab_tfidf/vocab_emb)*100:.1f}% smaller after lemmatization+stops")
print()

# Top 20 most frequent tokens per variant
tfidf_freq = Counter(all_tfidf_tokens).most_common(20)
print("Top 20 tokens (TF-IDF variant):")
for token, count in tfidf_freq:
    print(f"  {token:<20} : {count:,}")


=== Vocabulary Analysis ===
TF-IDF variant vocab size  : 41,664 unique tokens
Embedding variant vocab    : 46,255 unique tokens
Vocab reduction            : 9.9% smaller after lemmatization+stops

Top 20 tokens (TF-IDF variant):
  country              : 11,106
  policy               : 7,571
  development          : 6,328
  woman                : 5,617
  system               : 5,287
  water                : 5,250
  public               : 5,238
  social               : 5,131
  right                : 4,812
  health               : 4,725
  law                  : 4,699
  state                : 4,605
  international        : 4,586
  government           : 4,580
  education            : 4,535
  service              : 4,534
  national             : 4,530
  area                 : 4,385
  income               : 4,253
  per                  : 4,054


In [10]:
# Cell 10: N-gram preview 
# N-grams are generated by TfidfVectorizer in Step 4 automatically
# This cell just previews what bigrams look like from the training data

from sklearn.feature_extraction.text import CountVectorizer

print("=== N-gram Preview ===")
print("(Full N-gram features are built in Step 4 via TfidfVectorizer)")
print()

# Unigrams
cv1 = CountVectorizer(ngram_range=(1,1), max_features=10, stop_words="english")
cv1.fit(X_train_tfidf.fillna(""))
print("Top 10 unigrams:")
print(" ", list(cv1.vocabulary_.keys()))

# Bigrams
cv2 = CountVectorizer(ngram_range=(2,2), max_features=10)
cv2.fit(X_train_tfidf.fillna(""))
print("\nTop 10 bigrams:")
print(" ", list(cv2.vocabulary_.keys()))

# Trigrams
cv3 = CountVectorizer(ngram_range=(3,3), max_features=10)
cv3.fit(X_train_tfidf.fillna(""))
print("\nTop 10 trigrams:")
print(" ", list(cv3.vocabulary_.keys()))

print()
print("In Step 4, ngram_range=(1,2) will be used — unigrams + bigrams.")
print("Trigrams are usually too sparse to be useful.")


=== N-gram Preview ===
(Full N-gram features are built in Step 4 via TfidfVectorizer)

Top 10 unigrams:
  ['health', 'social', 'law', 'country', 'right', 'policy', 'woman', 'water', 'development', 'public']

Top 10 bigrams:
  ['health care', 'labour market', 'climate change', 'developing country', 'per cent', 'oecd country', 'human right', 'united state', 'international law', 'long term']

Top 10 trigrams:
  ['convention human right', 'sub saharan africa', 'play important role', 'per cent per', 'cent per cent', 'higher education institution', 'human right law', 'international human right', 'across oecd country', 'labour force participation']

In Step 4, ngram_range=(1,2) will be used — unigrams + bigrams.
Trigrams are usually too sparse to be useful.


In [11]:
# Cell 11: Save all preprocessed variants 

# TF-IDF variant
pd.DataFrame({"text": X_train_tfidf}).to_csv(
    CLEAN_DIR / "X_train_tfidf.csv", index=False)
pd.DataFrame({"text": X_test_tfidf}).to_csv(
    CLEAN_DIR / "X_test_tfidf.csv", index=False)
pd.DataFrame({"text": mahe_tfidf}).to_csv(
    CLEAN_DIR / "mahe_tfidf.csv", index=False)

# Embedding variant
pd.DataFrame({"text": X_train_emb}).to_csv(
    CLEAN_DIR / "X_train_emb.csv", index=False)
pd.DataFrame({"text": X_test_emb}).to_csv(
    CLEAN_DIR / "X_test_emb.csv", index=False)
pd.DataFrame({"text": mahe_emb}).to_csv(
    CLEAN_DIR / "mahe_emb.csv", index=False)

# BERT variant
pd.DataFrame({"text": X_train_bert}).to_csv(
    CLEAN_DIR / "X_train_bert.csv", index=False)
pd.DataFrame({"text": X_test_bert}).to_csv(
    CLEAN_DIR / "X_test_bert.csv", index=False)
pd.DataFrame({"text": mahe_bert}).to_csv(
    CLEAN_DIR / "mahe_bert.csv", index=False)

print("💾 Saved all preprocessed variants to data/clean/:")
print()
print("  TF-IDF variant (Steps 4):")
print("    X_train_tfidf.csv")
print("    X_test_tfidf.csv")
print("    mahe_tfidf.csv")
print()
print("  Embedding variant (Step 5):")
print("    X_train_emb.csv")
print("    X_test_emb.csv")
print("    mahe_emb.csv")
print()
print("  BERT variant (Step 7):")
print("    X_train_bert.csv")
print("    X_test_bert.csv")
print("    mahe_bert.csv")


💾 Saved all preprocessed variants to data/clean/:

  TF-IDF variant (Steps 4):
    X_train_tfidf.csv
    X_test_tfidf.csv
    mahe_tfidf.csv

  Embedding variant (Step 5):
    X_train_emb.csv
    X_test_emb.csv
    mahe_emb.csv

  BERT variant (Step 7):
    X_train_bert.csv
    X_test_bert.csv
    mahe_bert.csv


In [12]:
# Cell 12: Step 3 summary 
print("=" * 60)
print("  STEP 3 — PREPROCESSING SUMMARY")
print("=" * 60)
print()
print("  Three preprocessing variants produced:")
print()
print("  1. TF-IDF variant")
print("     Pipeline : clean → tokenize → remove stopwords → lemmatize")
print("     Used in  : Step 4 (TF-IDF + classifiers)")
print(f"     Vocab    : {vocab_tfidf:,} unique tokens")
print()
print("  2. Embedding variant")
print("     Pipeline : clean → tokenize → keep stopwords")
print("     Used in  : Step 5 (Word2Vec, GloVe)")
print(f"     Vocab    : {vocab_emb:,} unique tokens")
print()
print("  3. BERT variant")
print("     Pipeline : minimal clean → truncate to 512 chars")
print("     Used in  : Step 7 (Sentence-BERT, SciBERT)")
print(f"     Note     : BERT tokenizer handles everything else")
print()
print("  Domain stopwords added : 36 academic words removed")
print("  (study, analysis, result, paper, research, etc.)")
print()
print("✅ Step 3 complete → next: step4_baseline.ipynb")
print("=" * 60)


  STEP 3 — PREPROCESSING SUMMARY

  Three preprocessing variants produced:

  1. TF-IDF variant
     Pipeline : clean → tokenize → remove stopwords → lemmatize
     Used in  : Step 4 (TF-IDF + classifiers)
     Vocab    : 41,664 unique tokens

  2. Embedding variant
     Pipeline : clean → tokenize → keep stopwords
     Used in  : Step 5 (Word2Vec, GloVe)
     Vocab    : 46,255 unique tokens

  3. BERT variant
     Pipeline : minimal clean → truncate to 512 chars
     Used in  : Step 7 (Sentence-BERT, SciBERT)
     Note     : BERT tokenizer handles everything else

  Domain stopwords added : 36 academic words removed
  (study, analysis, result, paper, research, etc.)

✅ Step 3 complete → next: step4_baseline.ipynb
